# LLC Additivity on CMSP — Google Colab

**Setup**: Runtime → Change runtime type → **T4 GPU** (or better)

This notebook runs the full pipeline: train CMSP models, estimate data-restricted LLCs, and compute additivity defects and ratios for pairs and triplets of tasks.

## 0. Install & Setup

In [ ]:
# Mount Google Drive (optional — for persisting results across sessions)
SAVE_TO_DRIVE = True  # Set False to save locally (lost on runtime disconnect)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_ROOT = '/content/drive/MyDrive/llc_cmsp_results'
else:
    RESULTS_ROOT = '/content/results'

In [ ]:
# Install devinterp WITHOUT replacing Colab's pre-installed PyTorch
# (replacing it causes a torch._dynamo circular import error)
!pip install devinterp --no-deps -q
!pip install pyyaml tqdm pandas matplotlib -q

# Install devinterp's non-torch dependencies
!pip install nflows typing_extensions -q

# Clone your project repo (edit the URL after pushing to GitHub)
!git clone https://github.com/YOUR_USERNAME/llc-modularity.git /content/llc-modularity 2>/dev/null || echo 'Already cloned'

print('\n--- Restart the runtime now if this is your first install ---')
print('Go to: Runtime → Restart runtime, then re-run from the next cell onward.')

In [ ]:
import sys, os
sys.path.insert(0, '/content/llc-modularity')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU detected. LLC estimation will be very slow.')
    DEVICE = 'cpu'

In [ ]:
# Quick import check
from src.data import generate_cmsp_batch, make_subtask_indices, CMSPDataset
from src.model import make_mlp, count_parameters
from src.train import train_cmsp
from src.utils import plot_training_curves
from src.llc_estimation import estimate_llc, estimate_subtask_llcs
from src.additivity import (
    compute_additivity_defect, compute_full_additivity_defect,
    compute_triplet_additivity, enumerate_pair_triples,
    enumerate_triplet_quads, summarize_results,
)
from src.data import make_subtask_dataloaders, make_union_dataloaders, code_to_str
print('All imports OK')

## 1. Configuration

Edit the config dict below to control the experiment.

In [ ]:
from pathlib import Path

EXPERIMENT = 'B'  # 'A' = independent atomics, 'B' = composite curriculum

config = dict(
    m=4, k=4, n=16,
    width=128, depth=3,
    activation='relu', use_layernorm=False,
    samples_per_task=2000, lr=1e-3,
    dtype='float32', device=DEVICE,
    eval_every=500, checkpoint_every=0, checkpoint_steps=[],
)

if EXPERIMENT == 'A':
    config['task_codes'] = [[0], [1]]
    config['steps'] = 100_000
elif EXPERIMENT == 'B':
    config['task_codes'] = [[0], [1], [2], [3], [0, 1, 2, 3]]
    config['steps'] = 200_000

LLC_CONFIG = dict(
    num_chains=10, num_draws=500, num_burnin_steps=100,
    num_steps_bw_draws=1, learning_rate=1e-4,
    localization=0.0, seed=42,
)

LLC_SAMPLES_PER_CODE = 2000
LLC_BATCH_SIZE = 256
SEEDS = [0, 1, 2]

print(f'Experiment {EXPERIMENT}: codes={config["task_codes"]}')
print(f'Width={config["width"]}, depth={config["depth"]}, steps={config["steps"]}')
print(f'Seeds: {SEEDS}, LLC: {LLC_CONFIG["num_chains"]} chains x {LLC_CONFIG["num_draws"]} draws')

## 2. Train Models

In [ ]:
trained_models = {}

for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'Training seed {seed}')
    print(f'{"="*60}')

    cfg = {**config, 'seed': seed}
    save_dir = os.path.join(RESULTS_ROOT, f'exp_{EXPERIMENT}', f'seed_{seed}')
    os.makedirs(save_dir, exist_ok=True)

    results = train_cmsp(cfg, save_dir=save_dir, verbose=True)
    trained_models[seed] = results

    print(f'  Parameters: {results["n_parameters"]}')
    print('  Final losses (train / test):')
    for name in results['final_subtask_losses']:
        tr = results['final_subtask_losses'][name]
        te = results['final_test_subtask_losses'][name]
        print(f'    {name}: {tr:.6f} / {te:.6f}')

### 2a. Training Curves (Train & Test)

In [ ]:
import matplotlib.pyplot as plt

# Per-seed train vs test curves
for seed in SEEDS:
    print(f'Seed {seed}:')
    plot_training_curves(trained_models[seed], show=True)

# Overlay all seeds on one figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for seed in SEEDS:
    res = trained_models[seed]
    eval_steps = res['eval_steps']
    for name, losses in res['test_subtask_losses'].items():
        axes[0].plot(eval_steps[:len(losses)], losses, label=f'{name} (s{seed})', alpha=0.7)
    for name, losses in res['subtask_losses'].items():
        axes[1].plot(eval_steps[:len(losses)], losses, label=f'{name} (s{seed})', alpha=0.7)

axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].set_title('Test Loss (fixed dataset) \u2014 all seeds')
axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].legend(fontsize=6, ncol=2)

axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss')
axes[1].set_title('Train Loss (fresh samples) \u2014 all seeds')
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].legend(fontsize=6, ncol=2)

plt.tight_layout()
plt.show()

## 3. LLC Estimation

For each trained model, estimate the LLC using data restricted to each subtask.

**Loss scaling**: For a union of $k$ codes, the evaluate function is scaled by $k$ so that the effective loss = sum of per-code losses (not the average). This ensures $\hat\lambda(T_1 \cup T_2) = \hat\lambda(T_1) + \hat\lambda(T_2)$ for independent tasks.

**Codes estimated**: Individual task codes (atomics, composites, pairwise composites), all pairwise unions, and all 3-way unions.

In [ ]:
import pickle
from itertools import combinations

m = config['m']
k = config['k']
n = config['n']
subtask_indices = make_subtask_indices(m, k)
task_codes = config['task_codes']
dtype = torch.float32

# All individual codes: trained codes + pairwise composites for triplet analysis
individual_codes = list(task_codes)
atomic_codes = [c for c in task_codes if len(c) == 1]

for ci, cj in combinations(range(m), 2):
    pair_code = [ci, cj]
    if pair_code not in individual_codes:
        individual_codes.append(pair_code)

code_names = [code_to_str(c) for c in individual_codes]
print(f'Individual codes: {code_names}')

# Build all union groups (pairwise + triplet)
union_groups = {}

for i in range(len(individual_codes)):
    for j in range(i + 1, len(individual_codes)):
        label = f'{code_names[i]}\u222a{code_names[j]}'
        union_groups[label] = [individual_codes[i], individual_codes[j]]

for i in range(len(individual_codes)):
    for j in range(i + 1, len(individual_codes)):
        for kk in range(j + 1, len(individual_codes)):
            label = f'{code_names[i]}\u222a{code_names[j]}\u222a{code_names[kk]}'
            union_groups[label] = [individual_codes[i], individual_codes[j], individual_codes[kk]]

n_pair = len([u for u in union_groups if u.count('\u222a') == 1])
n_trip = len([u for u in union_groups if u.count('\u222a') == 2])
print(f'Union dataloaders: {n_pair} pairwise + {n_trip} triplet = {len(union_groups)} total')

In [ ]:
all_llc_results = {}

for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'LLC estimation \u2014 seed {seed}')
    print(f'{"="*60}')

    model = trained_models[seed]['model']

    # Per-code dataloaders
    per_code_loaders = make_subtask_dataloaders(
        n=n, m=m, subtask_indices=subtask_indices,
        task_codes=individual_codes,
        samples_per_code=LLC_SAMPLES_PER_CODE,
        batch_size=LLC_BATCH_SIZE,
        device=DEVICE, dtype=dtype,
        seed=LLC_CONFIG['seed'],
    )

    # Union dataloaders
    union_loaders = make_union_dataloaders(
        n=n, m=m, subtask_indices=subtask_indices,
        code_groups=union_groups,
        samples_per_code=LLC_SAMPLES_PER_CODE,
        batch_size=LLC_BATCH_SIZE,
        device=DEVICE, dtype=dtype,
        seed=LLC_CONFIG['seed'],
    )
    per_code_loaders.update(union_loaders)

    # Run LLC estimation (num_codes auto-inferred from \u222a in names)
    llc_results = estimate_subtask_llcs(
        model=model,
        subtask_dataloaders=per_code_loaders,
        device=DEVICE,
        verbose=True,
        **LLC_CONFIG,
    )

    all_llc_results[seed] = llc_results

    save_dir = os.path.join(RESULTS_ROOT, f'exp_{EXPERIMENT}', f'seed_{seed}')
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, 'llc_results.pkl'), 'wb') as f:
        pickle.dump(llc_results, f)

print('\nLLC estimation complete for all seeds.')

### 3a. LLC Summary (Individual Codes)

In [ ]:
import pandas as pd

rows = []
for seed, llc_res in all_llc_results.items():
    for name, r in llc_res.items():
        if '\u222a' not in name:
            rows.append({'seed': seed, 'subtask': name,
                         'llc_mean': r['llc_mean'], 'llc_std': r['llc_std']})

llc_df = pd.DataFrame(rows)
summary = llc_df.groupby('subtask').agg(
    llc_avg=('llc_mean', 'mean'),
    llc_sem=('llc_mean', 'std'),
).sort_index()
print('Individual code LLCs (averaged over seeds):')
print(summary.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(summary)), summary['llc_avg'], yerr=summary['llc_sem'],
       capsize=5, alpha=0.8, color='steelblue')
ax.set_xticks(range(len(summary)))
ax.set_xticklabels(summary.index, rotation=45, ha='right')
ax.set_ylabel(r'LLC ($\hat{\lambda}$)')
ax.set_title(f'Experiment {EXPERIMENT}: LLC per Individual Code (avg over {len(SEEDS)} seeds)')
plt.tight_layout()
plt.show()

### 3b. SGLD Trace Diagnostics

In [ ]:
import numpy as np

seed0 = SEEDS[0]
llc_res = all_llc_results[seed0]
subtasks_to_plot = [k for k in sorted(llc_res.keys()) if '\u222a' not in k][:4]

fig, axes = plt.subplots(1, len(subtasks_to_plot), figsize=(5*len(subtasks_to_plot), 4))
if len(subtasks_to_plot) == 1:
    axes = [axes]

for ax, name in zip(axes, subtasks_to_plot):
    trace = llc_res[name].get('loss_trace')
    if trace is not None:
        trace = np.array(trace) if not isinstance(trace, np.ndarray) else trace
        for chain_trace in trace:
            ax.plot(chain_trace, alpha=0.4, linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel('Draw')
    ax.set_ylabel('Loss')

plt.suptitle(f'SGLD Loss Traces (seed {seed0})', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Pairwise Additivity Defects & Ratios

For each pair $(T_1, T_2)$ of individual codes:
- **Defect**: $\delta = \hat\lambda(T_1) + \hat\lambda(T_2) - \hat\lambda(T_1 \cup T_2)$
- **Ratio**: $\rho = \hat\lambda(T_1 \cup T_2)\, /\, (\hat\lambda(T_1) + \hat\lambda(T_2))$

For independent tasks: $\delta \approx 0$, $\rho \approx 1$.

In [ ]:
all_pair_dfs = []

for seed, llc_res in all_llc_results.items():
    triples = enumerate_pair_triples(code_names, llc_res)
    if triples:
        df = compute_additivity_defect(llc_res, triples)
        df['seed'] = seed
        all_pair_dfs.append(df)

if all_pair_dfs:
    pair_df = pd.concat(all_pair_dfs, ignore_index=True)
    pair_agg = pair_df.groupby(['T1', 'T2']).agg(
        delta_avg=('delta', 'mean'), delta_sem=('delta', 'std'),
        ratio_avg=('ratio', 'mean'), ratio_sem=('ratio', 'std'),
    )
    print('Pairwise Additivity (averaged over seeds):')
    print(pair_agg.to_string())
else:
    print('No pairwise defects could be computed.')

In [ ]:
if all_pair_dfs:
    agg = pair_df.groupby(['T1', 'T2'])[['delta', 'ratio']].agg(['mean', 'std']).reset_index()
    labels = [f"{row[('T1','')]}+{row[('T2','')]}" for _, row in agg.iterrows()]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    # Defect plot
    deltas = agg[('delta', 'mean')].values
    delta_err = agg[('delta', 'std')].values
    colors = ['#2196F3' if d >= 0 else '#F44336' for d in deltas]
    ax1.bar(range(len(labels)), deltas, yerr=delta_err, capsize=3, color=colors, alpha=0.8)
    ax1.set_xticks(range(len(labels)))
    ax1.set_xticklabels(labels, rotation=90, fontsize=7)
    ax1.set_ylabel(r'Defect $\delta$')
    ax1.set_title('Pairwise Additivity Defect')
    ax1.axhline(y=0, color='black', linewidth=0.5)

    # Ratio plot
    ratios = agg[('ratio', 'mean')].values
    ratio_err = agg[('ratio', 'std')].values
    ax2.bar(range(len(labels)), ratios, yerr=ratio_err, capsize=3, color='steelblue', alpha=0.8)
    ax2.set_xticks(range(len(labels)))
    ax2.set_xticklabels(labels, rotation=90, fontsize=7)
    ax2.set_ylabel(r'Ratio $\rho = \lambda_{joint} / (\lambda_1 + \lambda_2)$')
    ax2.set_title('Pairwise Additivity Ratio')
    ax2.axhline(y=1, color='black', linestyle='--', linewidth=0.5)

    plt.tight_layout()
    plt.show()

## 5. Triplet Additivity Defects & Ratios

For each triple $(T_1, T_2, T_3)$ of individual codes:
- **Ratio**: $\rho = \hat\lambda(T_1 \cup T_2 \cup T_3)\, /\, (\hat\lambda(T_1) + \hat\lambda(T_2) + \hat\lambda(T_3))$

Interesting triplets with shared structure:
- $\{\{0\}, \{1\}, \{0,1\}\}$ — two atomics + their composite
- $\{\{0,1\}, \{1,2\}, \{0,2\}\}$ — three composites sharing atomic subtasks

In [ ]:
all_triplet_dfs = []

for seed, llc_res in all_llc_results.items():
    quads = enumerate_triplet_quads(code_names, llc_res)
    if quads:
        df = compute_triplet_additivity(llc_res, quads)
        df['seed'] = seed
        all_triplet_dfs.append(df)

if all_triplet_dfs:
    triplet_df = pd.concat(all_triplet_dfs, ignore_index=True)
    triplet_agg = triplet_df.groupby(['T1', 'T2', 'T3']).agg(
        delta_avg=('delta', 'mean'), delta_sem=('delta', 'std'),
        ratio_avg=('ratio', 'mean'), ratio_sem=('ratio', 'std'),
    )
    print(f'Triplet Additivity ({len(triplet_agg)} triplets, averaged over {len(SEEDS)} seeds):')
    print(triplet_agg.to_string())
else:
    print('No triplet defects could be computed.')

In [ ]:
if all_triplet_dfs:
    agg = triplet_df.groupby(['T1', 'T2', 'T3'])[['delta', 'ratio']].agg(['mean', 'std']).reset_index()
    labels = [f"{row[('T1','')]}+{row[('T2','')]}+{row[('T3','')]}" for _, row in agg.iterrows()]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Defect
    deltas = agg[('delta', 'mean')].values
    delta_err = agg[('delta', 'std')].values
    colors = ['#2196F3' if d >= 0 else '#F44336' for d in deltas]
    ax1.bar(range(len(labels)), deltas, yerr=delta_err, capsize=2, color=colors, alpha=0.8)
    ax1.set_xticks(range(len(labels)))
    ax1.set_xticklabels(labels, rotation=90, fontsize=5)
    ax1.set_ylabel(r'Defect $\delta$')
    ax1.set_title('Triplet Additivity Defect')
    ax1.axhline(y=0, color='black', linewidth=0.5)

    # Ratio
    ratios = agg[('ratio', 'mean')].values
    ratio_err = agg[('ratio', 'std')].values
    ax2.bar(range(len(labels)), ratios, yerr=ratio_err, capsize=2, color='steelblue', alpha=0.8)
    ax2.set_xticks(range(len(labels)))
    ax2.set_xticklabels(labels, rotation=90, fontsize=5)
    ax2.set_ylabel(r'Ratio $\rho = \lambda_{joint} / \sum \lambda_i$')
    ax2.set_title('Triplet Additivity Ratio')
    ax2.axhline(y=1, color='black', linestyle='--', linewidth=0.5)

    plt.tight_layout()
    plt.show()

### 5a. Highlight: Interesting Triplets

Filter for triplets with shared structure (atomics + their composite, or composites sharing an atomic).

In [ ]:
if all_triplet_dfs:
    interesting = []
    for _, row in triplet_agg.reset_index().iterrows():
        codes = [row['T1'], row['T2'], row['T3']]
        interesting.append({
            'triplet': ' + '.join(codes),
            'ratio': row['ratio_avg'],
            'ratio_err': row['ratio_sem'],
            'delta': row['delta_avg'],
            'delta_err': row['delta_sem'],
        })

    int_df = pd.DataFrame(interesting).sort_values('ratio')
    print('All triplet ratios (sorted by ratio):')
    for _, row in int_df.iterrows():
        flag = ' ***' if abs(row['ratio'] - 1.0) > 0.15 else ''
        print(f"  {row['triplet']:50s}  rho={row['ratio']:.3f}+/-{row['ratio_err']:.3f}  "
              f"delta={row['delta']:.3f}+/-{row['delta_err']:.3f}{flag}")
else:
    print('No triplet data available.')

## 6. Full Composite Defect (Experiment B)

In [ ]:
if EXPERIMENT == 'B':
    atomic_names = [code_to_str([i]) for i in range(config['m'])]
    composite_key = '\u222a'.join(atomic_names)

    for seed, llc_res in all_llc_results.items():
        if composite_key in llc_res:
            r = compute_full_additivity_defect(llc_res, atomic_names, composite_key)
            print(f"Seed {seed}: sum_atomic={r['sum_atomic_llc']:.3f}, "
                  f"composite={r['composite_llc']:.3f}, "
                  f"delta={r['delta']:.3f}+/-{r['delta_std']:.3f}, "
                  f"ratio={r['ratio']:.3f}+/-{r['ratio_std']:.3f}")
        else:
            print(f'Seed {seed}: composite key "{composite_key}" not in results')
            print(f'  Available: {sorted(llc_res.keys())[:10]}...')
else:
    print('Full composite defect only for Experiment B.')

## 7. Save All Results

In [ ]:
save_path = os.path.join(RESULTS_ROOT, f'exp_{EXPERIMENT}', 'summary.pkl')
os.makedirs(os.path.dirname(save_path), exist_ok=True)

summary_data = {
    'config': config,
    'llc_config': LLC_CONFIG,
    'seeds': SEEDS,
    'llc_df': llc_df,
    'pair_df': pair_df if all_pair_dfs else None,
    'triplet_df': triplet_df if all_triplet_dfs else None,
}

with open(save_path, 'wb') as f:
    pickle.dump(summary_data, f)

print(f'Summary saved to {save_path}')
if SAVE_TO_DRIVE:
    print('Results are on Google Drive and will persist across sessions.')

---

## Appendix: Quick Smoke Test

Uncomment to validate the pipeline quickly (~2 min on GPU) before a full run.

In [ ]:
# quick_config = dict(
#     m=2, k=3, n=6, width=32, depth=2,
#     activation='relu', use_layernorm=False,
#     task_codes=[[0], [1]],
#     samples_per_task=500, steps=5000, lr=1e-3,
#     seed=0, dtype='float32', device=DEVICE,
#     eval_every=500, checkpoint_every=0, checkpoint_steps=[],
# )
#
# res = train_cmsp(quick_config, verbose=True)
# plot_training_curves(res, show=True)
#
# si = make_subtask_indices(2, 3)
# loaders = make_subtask_dataloaders(
#     n=6, m=2, subtask_indices=si, task_codes=[[0], [1]],
#     samples_per_code=500, batch_size=128, device=DEVICE, dtype=torch.float32, seed=42,
# )
# union_loaders = make_union_dataloaders(
#     n=6, m=2, subtask_indices=si,
#     code_groups={'{0}\u222a{1}': [[0], [1]]},
#     samples_per_code=500, batch_size=128, device=DEVICE, dtype=torch.float32, seed=42,
# )
# loaders.update(union_loaders)
#
# llc_res = estimate_subtask_llcs(
#     model=res['model'], subtask_dataloaders=loaders,
#     device=DEVICE, num_chains=3, num_draws=50,
#     num_burnin_steps=20, learning_rate=1e-4, seed=42, verbose=True,
# )
#
# df = compute_additivity_defect(llc_res, [('{0}', '{1}', '{0}\u222a{1}')])
# print(summarize_results(llc_res, df))